# Multi-Agent Stock Trading Comparison (PPO vs DDPG vs SAC)

This notebook follows the training and testing workflow of `ppo_stock.ipynb`, then compares three RL agents (`PPO`, `DDPG`, `SAC`) against `DJI` benchmark on the same test horizon.

Outputs:
- Trained checkpoints for all three agents
- Test account value CSV files
- One comparison chart in a style similar to your reference figure

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import torch
import yfinance as yf
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Compatibility patches used in existing notebooks/scripts
if not hasattr(torch, '_original_load'):
    torch._original_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return torch._original_load(*args, **kwargs)

torch.load = patched_torch_load

if not hasattr(yf, '_original_download'):
    yf._original_download = yf.download

def patched_download(*args, **kwargs):
    kwargs.pop('proxy', None)
    kwargs.setdefault('progress', False)
    kwargs.setdefault('auto_adjust', False)
    return yf._original_download(*args, **kwargs)

yf.download = patched_download

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.config_tickers import DOW_30_TICKER
from finrl.meta.preprocessor.preprocessors import FeatureEngineer
from elegantrl.agents import AgentPPO, AgentDDPG, AgentSAC
from elegantrl.train.run import train_agent
from env_wrapper.ppo_env_wrapper import ElegantFinRLWrapper

try:
    from elegantrl.train.config import Config as Arguments
except ImportError:
    from elegantrl.train.config import Arguments

print('[OK] Imports complete')

[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[OK] Imports complete


In [2]:
ALL_INDICATORS = ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30', 'close_60']

def prepare_data(cache_file='./data/stock_dow30_cache.csv',
                 TRAIN_START_DATE='2010-01-01',
                 TEST_END_DATE='2023-12-30'):
    os.makedirs(os.path.dirname(cache_file), exist_ok=True)

    if os.path.exists(cache_file):
        print(f'[INFO] Loading cache: {cache_file}')
        try:
            df = pd.read_csv(cache_file)
            df = df.drop_duplicates(subset=['date', 'tic'])
            if len(df) > 0:
                print(f'[OK] Cache rows={len(df)}, range={df["date"].min()} -> {df["date"].max()}')
                return df
        except Exception as e:
            print(f'[WARN] Cache load failed: {e}; redownloading')

    print('[STEP] Downloading raw DOW30 data...')
    df = YahooDownloader(
        start_date=TRAIN_START_DATE,
        end_date=TEST_END_DATE,
        ticker_list=DOW_30_TICKER
    ).fetch_data()

    print('[STEP] Adding technical indicators...')
    tech_for_fe = ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']

    fe = FeatureEngineer(
        use_technical_indicator=True,
        tech_indicator_list=tech_for_fe,
        use_vix=False,
        use_turbulence=False,
        user_defined_feature=False
    )
    df = fe.preprocess_data(df)

    if 'close_30_sma' in df.columns:
        df = df.rename(columns={'close_30_sma': 'close_30'})
    if 'close_60_sma' in df.columns:
        df = df.rename(columns={'close_60_sma': 'close_60'})

    df.to_csv(cache_file, index=False)
    print(f'[OK] Cached to {cache_file}')
    return df

def split_train_test(df,
                     TRAIN_START_DATE='2010-01-07',
                     TRAIN_END_DATE='2023-10-24',
                     TEST_START_DATE1='2023-10-25',
                     TEST_END_DATE1='2023-11-14',
                     TEST_START_DATE2='2023-11-15',
                     TEST_END_DATE2='2023-11-22'):
    df['date'] = pd.to_datetime(df['date'])

    df_train = df[(df['date'] >= TRAIN_START_DATE) & (df['date'] <= TRAIN_END_DATE)].copy()
    df_test1 = df[(df['date'] >= TEST_START_DATE1) & (df['date'] <= TEST_END_DATE1)].copy()
    df_test2 = df[(df['date'] >= TEST_START_DATE2) & (df['date'] <= TEST_END_DATE2)].copy()

    return df_train, df_test1, df_test2

df = prepare_data()
df_train, df_test1, df_test2 = split_train_test(df)

print(f'[INFO] train rows={len(df_train)}, test1 rows={len(df_test1)}, test2 rows={len(df_test2)}')

[INFO] Loading cache: ./data/stock_dow30_cache.csv
[OK] Cache rows=29174, range=2020-01-02 -> 2023-12-29
[INFO] train rows=27840, test1 rows=435, test2 rows=174


In [3]:
def add_day_index(df_part):
    out = df_part.copy()
    dates = sorted(out['date'].unique())
    date_map = {d: i for i, d in enumerate(dates)}
    out['day'] = out['date'].map(date_map)
    out.set_index('day', inplace=True, drop=False)
    return out

df_train = add_day_index(df_train)
df_test1 = add_day_index(df_test1)
df_test2 = add_day_index(df_test2)

stock_dimension = len(df_train['tic'].unique())
state_dim = 1 + stock_dimension * (len(ALL_INDICATORS) + 2)

BASE_ENV_PARAMS = {
    'env_name': 'FinRL_Stock_Train',
    'stock_dim': stock_dimension,
    'hmax': 100,
    'initial_amount': 1_000_000,
    'num_stock_shares': [0] * stock_dimension,
    'buy_cost_pct': [0.001] * stock_dimension,
    'sell_cost_pct': [0.001] * stock_dimension,
    'reward_scaling': 1e-4,
    'state_space': state_dim,
    'action_space': stock_dimension,
    'tech_indicator_list': ALL_INDICATORS,
    'state_dim': state_dim,
    'action_dim': stock_dimension,
    'if_discrete': False,
    'target_return': 10.0,
}

print(f'[INFO] stock_dimension={stock_dimension}, state_dim={state_dim}')

[INFO] stock_dimension=29, state_dim=291


In [4]:
def setup_args(agent_class, env_args, cwd_path):
    args = Arguments(agent_class=agent_class, env_class=ElegantFinRLWrapper)
    args.env_args = env_args
    args.env_name = env_args['env_name']

    args.net_dims = (128, 64)
    args.state_dim = env_args['state_dim']
    args.action_dim = env_args['action_dim']
    args.if_discrete = env_args['if_discrete']

    args.learning_rate = 1e-4
    args.batch_size = 128

    args.target_step = 2000
    args.break_step = 40000

    args.worker_num = 1
    args.eval_proc_num = 0
    args.if_use_multi_processing = False
    args.eval_gap = 500
    args.save_gap = 200

    args.if_save = True
    args.if_overwrite_save = True

    args.cwd = cwd_path
    args.if_remove = False
    return args

def run_inference(agent_class, args, test_df, env_params):
    test_env_params = dict(env_params)
    test_env_params['df'] = test_df

    env = ElegantFinRLWrapper(**test_env_params)
    agent = agent_class(args.net_dims, args.state_dim, args.action_dim)
    agent.save_or_load_agent(args.cwd, if_save=False)
    agent.act.eval()

    res = env.reset()
    state = res[0] if isinstance(res, tuple) else res

    done = False
    while not done:
        s_tensor = torch.as_tensor((state,), dtype=torch.float32, device=agent.device)
        with torch.no_grad():
            action = agent.act(s_tensor).detach().cpu().numpy()[0]

        step_res = env.step(action)
        if len(step_res) == 5:
            state, reward, term, trunc, _ = step_res
            done = bool(term or trunc)
        else:
            state, reward, done, _ = step_res

    return env.env.save_asset_memory()

def train_and_test_one(name, agent_class, train_df, test_df1, test_df2, base_env_params):
    print('\n' + '=' * 80)
    print(f'[TRAIN] {name}')
    print('=' * 80)

    env_params = dict(base_env_params)
    env_params['env_name'] = f'FinRL_{name}_Train'
    env_params['df'] = train_df

    ckpt_dir = f'./checkpoints/stock/{name.lower()}_full_train'
    os.makedirs(ckpt_dir, exist_ok=True)

    args = setup_args(agent_class, env_params, ckpt_dir)
    train_agent(args)

    print(f'[TEST] {name} inference on test windows')
    pred1 = run_inference(agent_class, args, test_df1, env_params)
    pred2 = run_inference(agent_class, args, test_df2, env_params)

    pred1['date'] = pd.to_datetime(pred1['date'])
    pred2['date'] = pd.to_datetime(pred2['date'])
    merged = pd.concat([pred1, pred2], axis=0, ignore_index=True)
    merged = merged.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)

    out_dir = './logs/stock'
    os.makedirs(out_dir, exist_ok=True)
    out_file = f'{out_dir}/{name.lower()}_test_results_tot.csv'
    merged.to_csv(out_file, index=False)
    print(f'[OK] Saved {name} test results -> {out_file}')

    return merged

In [5]:
# Set to False if you already have result CSVs and only want to re-plot
RUN_TRAINING = True

AGENTS = {
    'PPO': AgentPPO,
    'DDPG': AgentDDPG,
    'SAC': AgentSAC,
}

results = {}

if RUN_TRAINING:
    for name, agent_class in AGENTS.items():
        results[name] = train_and_test_one(
            name=name,
            agent_class=agent_class,
            train_df=df_train,
            test_df1=df_test1,
            test_df2=df_test2,
            base_env_params=BASE_ENV_PARAMS,
        )
else:
    for name in AGENTS.keys():
        f = f'./logs/stock/{name.lower()}_test_results_tot.csv'
        if not os.path.exists(f):
            raise FileNotFoundError(f'Cannot find {f}, please train first or set RUN_TRAINING=True')
        tmp = pd.read_csv(f)
        tmp['date'] = pd.to_datetime(tmp['date'])
        tmp = tmp.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)
        results[name] = tmp

print('[OK] Result data prepared for PPO/DDPG/SAC')


[TRAIN] PPO
| train_agent_multiprocessing() with GPU_ID 0
| Arguments Keep cwd: ./checkpoints/stock/ppo_full_train
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
[✓] PPO Environment Wrappers imported successfully
[✓] FastContinuousCryptoEnv & ElegantFinRLWrapper classes defined!
| Evaluator:
| `step`: Number of samples, or total training steps, or running times of `env.step()`.
| `time`: Time spent from the start of training to this moment.
| `avgR`: Average value of cumulative rewards, which is the sum of rewards in an episode.
| `stdR`: Standard dev of cumulative rewards, which is the sum of rewards in an episode.
| `avgS`: Average of steps in an episode.
| `objC`: Objective of Cri

In [7]:
def calculate_metrics(df, value_col='account_value'):
    x = df[value_col].dropna()
    daily = x.pct_change().dropna()

    ret = float(x.iloc[-1] / x.iloc[0] - 1.0) if len(x) > 1 else 0.0
    shp = float((252 ** 0.5) * (daily.mean() / daily.std())) if daily.std() != 0 else 0.0

    running_max = x.cummax()
    dd = (x - running_max) / running_max
    mdd = float(dd.min()) if len(dd) else 0.0

    return ret, shp, mdd


def build_dji_curve(reference_df, initial_capital):
    start_date = reference_df['date'].min().strftime('%Y-%m-%d')
    end_date = (reference_df['date'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

    bench = yf.download('^DJI', start=start_date, end=end_date, progress=False)
    if isinstance(bench.columns, pd.MultiIndex):
        bench.columns = bench.columns.get_level_values(0)

    bench = bench.reset_index()
    if 'Date' not in bench.columns and 'index' in bench.columns:
        bench = bench.rename(columns={'index': 'Date'})

    bench['Date'] = pd.to_datetime(bench['Date'])

    merged = pd.merge(
        reference_df[['date']].copy(),
        bench[['Date', 'Close']],
        left_on='date',
        right_on='Date',
        how='left'
    )
    merged['Close'] = merged['Close'].ffill().bfill()
    merged['account_value'] = (merged['Close'] / merged['Close'].iloc[0]) * initial_capital

    return merged[['date', 'account_value']]


def plot_comparison_for_period(start_date, end_date, tag):
    period_results = {}

    for k in ['PPO', 'DDPG', 'SAC']:
        tmp = results[k].copy()
        tmp['date'] = pd.to_datetime(tmp['date'])
        tmp = tmp[(tmp['date'] >= start_date) & (tmp['date'] <= end_date)].copy()
        tmp = tmp.drop_duplicates(subset=['date']).sort_values('date').reset_index(drop=True)
        period_results[k] = tmp

    # Use PPO date index as common axis for all curves in this period
    base = period_results['PPO'][['date']].copy()
    for k in ['PPO', 'DDPG', 'SAC']:
        period_results[k] = pd.merge(base, period_results[k][['date', 'account_value']], on='date', how='left')
        period_results[k]['account_value'] = period_results[k]['account_value'].ffill().bfill()

    initial_capital = float(period_results['PPO']['account_value'].iloc[0])
    period_results['DJI'] = build_dji_curve(period_results['PPO'], initial_capital)

    metrics = {}
    for k in ['SAC', 'DDPG', 'PPO', 'DJI']:
        r, s, m = calculate_metrics(period_results[k], 'account_value')
        metrics[k] = {'ret': r, 'shp': s, 'mdd': m}

    plt.style.use('seaborn-v0_8-darkgrid')
    fig = plt.figure(figsize=(16, 9))
    ax = plt.gca()

    ax.plot(period_results['SAC']['date'], period_results['SAC']['account_value'], color='crimson', linewidth=2.2, label='AI SAC Portfolio')
    ax.plot(period_results['DDPG']['date'], period_results['DDPG']['account_value'], color='dodgerblue', linewidth=2.0, label='AI DDPG Portfolio')
    ax.plot(period_results['PPO']['date'], period_results['PPO']['account_value'], color='forestgreen', linewidth=2.0, label='AI PPO Portfolio')
    ax.plot(period_results['DJI']['date'], period_results['DJI']['account_value'], color='slategray', linewidth=2.0, linestyle='--', label='Dow Jones (Benchmark)')

    ax.set_title(f'Deep Reinforcement Learning Trading Comparison ({tag})', fontsize=22, fontweight='bold')
    ax.set_xlabel('Date', fontsize=16)
    ax.set_ylabel('Portfolio Value ($)', fontsize=16)
    ax.tick_params(axis='both', labelsize=12)

    box_text = '\n'.join([
        f'SAC  Ret: {metrics["SAC"]["ret"]*100:+.2f}% | Shp: {metrics["SAC"]["shp"]:.2f} | MDD: {metrics["SAC"]["mdd"]*100:+.2f}%',
        f'DDPG Ret: {metrics["DDPG"]["ret"]*100:+.2f}% | Shp: {metrics["DDPG"]["shp"]:.2f} | MDD: {metrics["DDPG"]["mdd"]*100:+.2f}%',
        f'PPO  Ret: {metrics["PPO"]["ret"]*100:+.2f}% | Shp: {metrics["PPO"]["shp"]:.2f} | MDD: {metrics["PPO"]["mdd"]*100:+.2f}%',
        f'DJI  Ret: {metrics["DJI"]["ret"]*100:+.2f}% | Shp: {metrics["DJI"]["shp"]:.2f} | MDD: {metrics["DJI"]["mdd"]*100:+.2f}%',
    ])

    ax.text(
        0.02, 0.96, box_text,
        transform=ax.transAxes,
        fontsize=15,
        va='top',
        family='monospace',
        bbox=dict(boxstyle='round', facecolor='whitesmoke', edgecolor='gray', alpha=0.95)
    )

    ax.legend(loc='upper right', fontsize=15, framealpha=0.95)
    plt.tight_layout()

    os.makedirs('./logs/stock', exist_ok=True)
    out_fig = f'./logs/stock/rl_trading_comparison_{tag.lower().replace(" ", "_")}.png'
    plt.savefig(out_fig, dpi=300, bbox_inches='tight')
    print(f'[OK] Saved figure -> {out_fig}')

    plt.show()

    return pd.DataFrame([
        {'Strategy': k, 'Return(%)': metrics[k]['ret'] * 100, 'Sharpe': metrics[k]['shp'], 'MaxDrawdown(%)': metrics[k]['mdd'] * 100}
        for k in ['SAC', 'DDPG', 'PPO', 'DJI']
    ])


# Plot test set 1 and test set 2 separately based on split ranges
test1_start = pd.to_datetime(df_test1['date']).min()
test1_end = pd.to_datetime(df_test1['date']).max()
test2_start = pd.to_datetime(df_test2['date']).min()
test2_end = pd.to_datetime(df_test2['date']).max()

summary_test1 = plot_comparison_for_period(test1_start, test1_end, 'Test Set 1')
summary_test2 = plot_comparison_for_period(test2_start, test2_end, 'Test Set 2')

print('\n[SUMMARY] Test Set 1')
display(summary_test1)
print('\n[SUMMARY] Test Set 2')
display(summary_test2)

[OK] Saved figure -> ./logs/stock/rl_trading_comparison_test_set_1.png
[OK] Saved figure -> ./logs/stock/rl_trading_comparison_test_set_2.png

[SUMMARY] Test Set 1


,Strategy,Return(%),Sharpe,MaxDrawdown(%)
0,SAC,1.489776,7.951001,-0.228764
1,DDPG,6.770526,8.076168,-1.517909
2,PPO,5.382748,7.425870,-1.018272
3,DJI,5.423699,6.890874,-1.871719



[SUMMARY] Test Set 2


,Strategy,Return(%),Sharpe,MaxDrawdown(%)
0,SAC,0.109698,8.418919,-0.021933
1,DDPG,1.050643,6.511682,-0.260894
2,PPO,0.348776,2.834774,-0.499235
3,DJI,0.805403,6.983815,-0.178515
